# Original baseline: ten ratings

Refit unigram TF-IDF and multinomial logistic regression on development data. Evaluate exact ratings and near misses on the original fixed test split.

Run from the repository root.

# Final baseline: review-only rating prediction, 80/20 refit

在先前约64/16/20划分上，validation已经用于选择 regParam=0.1。本次把原 train 和 validation 合并为 development（约80%），重新拟合词表、IDF和分类器；固定原 test（约20%），输出最终评估。只训练预选的0.1，不用test调整。

原测试集与其他实验已在项目探索中被查看，不能宣称全新盲测。组员清洗规则在clean_data.csv中已经实施：原始 review/rating distinct，同原始review不同rating的不同评分平均并取整，112,326条有效记录。模型预测1–10评分分布，不是情感标签。


In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, time, json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "clean_data.csv").is_file():
    raise FileNotFoundError("Place the archived clean_data.csv in the repository root.")
CLEAN_CSV = PROJECT_DIR / "clean_data.csv"
EXPECTED_CSV_SHA256 = "a74e8d67a9542a068f05823afd5c1fb34208fff1716e652ec6906485f891e911"
REFERENCE_DIR = PROJECT_DIR / "baseline_results" / "rating_teammate_policy_20261106_155015_111148"
SEED = 42
N_CLASSES = 10
CLASS_NAMES = [str(i) for i in range(1, 11)]
CLASS_TICKS = CLASS_NAMES
REG_PARAM = 0.1  # 固定之前在validation选出的参数
VOCAB_SIZE = 30000
MIN_DF = 2.0
MAX_ITER = 150
RUN_DIR = PROJECT_DIR / "baseline_results" / datetime.now().strftime("rating_teammate_refit80_%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
spark = (SparkSession.builder.appName("Rating_teammate_refit80")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable).getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("输出目录：", RUN_DIR)
print("Python:", platform.python_version(), "Spark:", pyspark.__version__)


## 1. 读取已清洗数据并核对来源

In [ ]:
schema = StructType([
    StructField("review", StringType()),StructField("rating",IntegerType()),
    StructField("filtered_words",StringType())])
if not CLEAN_CSV.exists():
    raise FileNotFoundError(CLEAN_CSV)
clean_hash = hashlib.sha256(CLEAN_CSV.read_bytes()).hexdigest()
assert clean_hash == EXPECTED_CSV_SHA256, "clean_data.csv与已核对版本不同，请先检查来源和行数"
cleanData = (spark.read.schema(schema).option("header",True).option("multiLine",True)
    .option("quote",'"').option("escape",'"').option("encoding","UTF-8")
    .option("mode","FAILFAST").csv(str(CLEAN_CSV))
    .withColumn("filtered_words",F.from_json("filtered_words",ArrayType(StringType()))))
valid = F.coalesce(F.col("review").isNotNull() & (F.length("review")>0)
    & F.col("rating").between(1,10) & (F.size("filtered_words")>0), F.lit(False))
assert cleanData.filter(~valid).limit(1).count()==0
assert cleanData.count()==112326
# CSV没有原始uniqueID；构造仅用于审计/导出用的稳定行键，不作模型输入。
cleanData = (cleanData.withColumn("review_original",F.col("review"))
    .withColumn("uniqueID",F.sha2(F.concat_ws("|",F.col("review"),F.col("rating").cast("string")),256))
    .withColumn("text_group_id",F.sha2(F.to_json("filtered_words"),256)))
print("组员规则清洗后的有效行数：",cleanData.count())


## 2. 固定原测试集，合并训练与验证

哈希规则沿用上一版。下面核对测试集中每个（文本组，评分）的计数，考虑清洗后可能相同的多条记录；不使用测试评分选择参数。

In [ ]:
model_data = (cleanData.withColumn("label", F.col("rating").cast("double") - 1.0)
    .withColumn("bucket",F.pmod(F.xxhash64("text_group_id",F.lit(SEED)),F.lit(10000)))
    .withColumn("split",F.when(F.col("bucket") < 8000,"development").otherwise("test"))
    .drop("bucket").cache())
development=model_data.filter(F.col("split")=="development").cache()
test=model_data.filter(F.col("split")=="test").cache()
assert model_data.groupBy("text_group_id").agg(F.countDistinct("split").alias("n")).filter("n>1").count()==0
assert development.count()==89912 and test.count()==22414
for name,df in [("development",development),("test",test)]:
    labels={int(r.label) for r in df.select("label").distinct().collect()}
    assert labels==set(range(10)),f"{name}缺少类别"
manifest=model_data.select("text_group_id","rating","label","split").toPandas()
reference=pd.read_csv(REFERENCE_DIR / "split_manifest.csv")
cols=["text_group_id","rating","label"]
# CSV reads rating as int64; Spark may return int32. Normalize key types before grouping.
def test_record_counts(frame):
    keys=frame.loc[frame.split == "test", cols].copy()
    if keys.isna().any().any():
        raise ValueError("测试集记录键含缺失值")
    keys["text_group_id"]=keys["text_group_id"].astype("string")
    for column in ["rating", "label"]:
        numeric=pd.to_numeric(keys[column], errors="raise")
        if not (numeric == np.floor(numeric)).all():
            raise ValueError(f"{column} 应为整数")
        keys[column]=numeric.astype("int64")
    return keys.groupby(cols,dropna=False).size().sort_index()
old_counts=test_record_counts(reference)
new_counts=test_record_counts(manifest)
pd.testing.assert_series_equal(old_counts,new_counts,check_exact=True)
manifest.to_csv(RUN_DIR/"split_manifest.csv",index=False)
distribution=model_data.groupBy("split","rating").count().orderBy("split","rating").toPandas()
distribution.to_csv(RUN_DIR/"split_distribution.csv",index=False)
display(distribution.pivot(index="rating",columns="split",values="count"))
print("development:",development.count(),"test:",test.count(),"测试集与原实验一致")


## 3. 在完整 development 集合拟合 unigram TF-IDF 和 LR

词表及IDF在 development 拟合，不使用 test 信息。

In [ ]:
feature_model=Pipeline(stages=[CountVectorizer(inputCol="filtered_words",outputCol="term_counts",vocabSize=VOCAB_SIZE,minDF=MIN_DF,binary=False),
    IDF(inputCol="term_counts",outputCol="features")]).fit(development)
development_features=feature_model.transform(development).cache()
test_features=feature_model.transform(test).cache()
start=time.perf_counter()
lr=LogisticRegression(featuresCol="features",labelCol="label",family="multinomial",regParam=REG_PARAM,
    elasticNetParam=0.0,maxIter=MAX_ITER,standardization=True,tol=1e-6)
model=lr.fit(development_features)
iterations=int(model.summary.totalIterations)
print("训练秒数:",round(time.perf_counter()-start,2),"迭代次数:",iterations,"词表大小:",len(feature_model.stages[0].vocabulary))
if iterations>=MAX_ITER: print("注意：达到迭代上限，需检查收敛。")
(RUN_DIR/"feature_metadata.json").write_text(json.dumps({"vocabulary":feature_model.stages[0].vocabulary},ensure_ascii=False),encoding="utf-8")
np.savez_compressed(RUN_DIR/"parameters.npz",idf=feature_model.stages[1].idf.toArray(),coefficients=model.coefficientMatrix.toArray(),intercept=model.interceptVector.toArray())
pd.DataFrame({"iteration":range(len(model.summary.objectiveHistory)),"objective":model.summary.objectiveHistory}).to_csv(RUN_DIR/"objective_history.csv",index=False)


## 4. 评价固定测试集

同时输出十分类 Accuracy、每类 Precision/Recall/F1、整数预测±1/±2命中率以及连续期望评分的 MAE/RMSE。

In [ ]:
def metrics_from_matrix(cm):
    cm = np.asarray(cm, dtype=np.int64)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    tp = np.diag(cm)
    precision = np.divide(tp, predicted, out=np.zeros(N_CLASSES), where=predicted != 0)
    recall = np.divide(tp, support, out=np.zeros(N_CLASSES), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(N_CLASSES), where=precision + recall != 0)
    n = int(cm.sum())
    if n == 0:
        raise ValueError('评估集合为空')
    weights = support / n
    summary = {'n': n, 'accuracy': float(tp.sum() / n), 'precision_macro': float(precision.mean()), 'recall_macro': float(recall.mean()), 'f1_macro': float(f1.mean()), 'precision_weighted': float(precision @ weights), 'recall_weighted': float(recall @ weights), 'f1_weighted': float(f1 @ weights)}
    per_class = pd.DataFrame({'class_id': np.arange(N_CLASSES), 'class_name': CLASS_NAMES, 'precision': precision, 'recall': recall, 'f1': f1, 'support': support})
    return (summary, per_class)

def rating_decisions(probabilities):
    p = np.asarray(probabilities,dtype=float)
    if p.ndim != 2 or p.shape[1] != 10 or not np.isfinite(p).all():
        raise ValueError("预测概率形状或数值无效")
    if (p < 0).any() or not np.allclose(p.sum(axis=1),1.0,atol=1e-6):
        raise ValueError("评分概率不满足概率分布要求")
    return {
        "map":p.argmax(axis=1)+1,
        "expected":p @ np.arange(1,11,dtype=float),
        "median":(np.cumsum(p,axis=1) >= 0.5).argmax(axis=1)+1
    }

def evaluate_rating_frame(predictions):
    local = predictions.select("uniqueID","text_group_id","rating",vector_to_array("probability").alias("probability")).toPandas()
    if len(local)==0:
        raise ValueError("评估集为空")
    y=local.rating.to_numpy(dtype=int)
    decisions=rating_decisions(np.stack(local.probability.to_numpy()))
    summary={"n":len(local)}
    for decision, values in decisions.items():
        error=np.abs(values-y)
        summary.update({"mae_"+decision:float(error.mean()),
                        "rmse_"+decision:float(np.sqrt(np.mean((values-y)**2))),
                        "within_1_"+decision:float(np.mean(error <= 1.0+1e-10)),
                        "within_2_"+decision:float(np.mean(error <= 2.0+1e-10))})
        if decision != "expected":
            summary["exact_accuracy_"+decision]=float(np.mean(values==y))
        local["predicted_"+decision]=values
    cm=np.zeros((10,10),dtype=np.int64)
    np.add.at(cm,(y-1,decisions["map"]-1),1)
    classification, report=metrics_from_matrix(cm)
    summary.update({"classification_"+k:v for k,v in classification.items() if k!='n'})
    report["predicted_count"]=cm.sum(axis=0)
    local["absolute_error_expected"]=np.abs(local.predicted_expected-y)
    local["probability"]=local.probability.map(lambda x:json.dumps([float(v) for v in x]))
    return summary,report,cm,local


In [ ]:
summary,report,cm,predictions=evaluate_rating_frame(model.transform(test_features))
(RUN_DIR/"final_test_metrics.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
report.to_csv(RUN_DIR/"final_test_classification_report.csv",index=False)
pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(RUN_DIR/"final_test_map_confusion.csv")
predictions.to_csv(RUN_DIR/"final_test_predictions.csv",index=False)
print("accuracy",round(summary["exact_accuracy_map"],4),"±1:",round(summary["within_1_map"],4),"±2:",round(summary["within_2_map"],4),"expected MAE:",round(summary["mae_expected"],4))
display(pd.DataFrame([summary]))
display(report)
fig,ax=plt.subplots(figsize=(8,7));im=ax.imshow(cm,cmap="Blues")
ax.set(xticks=range(10),yticks=range(10),xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,xlabel="Predicted rating",ylabel="Observed rating",title="Test confusion matrix")
for i in range(10):
    for j in range(10):ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=6,color="white" if cm[i,j]>cm.max()/2 else "black")
fig.colorbar(im,ax=ax);fig.tight_layout();fig.savefig(RUN_DIR/"test_confusion_matrix.png",dpi=150);plt.show();plt.close(fig)
config={"model":"TF-IDF + multinomial logistic regression","data_sha256":clean_hash,"seed":SEED,
    "split":"same prior test; old train+validation merged into development","development_n":89912,"test_n":22414,
    "parameter_selection":"regParam=0.1 selected earlier by validation expected MAE","reg_param":REG_PARAM,
    "vocab_size_limit":VOCAB_SIZE,"actual_vocab_size":len(feature_model.stages[0].vocabulary),"min_df":MIN_DF,"max_iter":MAX_ITER,"iterations":iterations,
    "test_caveat":"Test was previously viewed during exploration; not an untouched blind test",
    "prior_run":str(REFERENCE_DIR),"spark":pyspark.__version__}
(RUN_DIR/"experiment_config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")
print("输出目录:",RUN_DIR)
